# Advanced Problems: Python Lists vs Tuples

This notebook contains advanced problems with solutions on Python lists vs tuples, focusing on immutability, bytecode, constant folding, shallow copying, aliasing, memory behavior, amortized list growth, performance measurement, and API design.

## How to use this notebook

- Read each problem first.
- Run the optional setup/code cells.
- Attempt the problem before opening the solution section.
- Use the included checks to validate your reasoning.


## Setup

The problems use only the Python standard library.

In [1]:
from dis import dis
from timeit import timeit
import sys
import copy
from collections import namedtuple


## Problem 1 — Predict CPython Bytecode for Constant and Non-Constant Containers

**Difficulty:** Advanced

Consider the following expressions:

```python
(1, 2, 3, 'x')
[1, 2, 3, 'x']
(len, 1, 2)
[len, 1, 2]
([1, 2], 3, 4)
```

For each expression:

1. Predict whether CPython can load the container as a single constant.
2. Predict whether a `BUILD_TUPLE` or `BUILD_LIST` operation is needed.
3. Explain why the answer changes when an element is not a compile-time constant.


In [2]:
expressions = [
    "(1, 2, 3, 'x')",
    "[1, 2, 3, 'x']",
    "(len, 1, 2)",
    "[len, 1, 2]",
    "([1, 2], 3, 4)",
]

for expr in expressions:
    print(f"\nExpression: {expr}")
    dis(compile(expr, '<string>', 'eval'))



Expression: (1, 2, 3, 'x')
  0           RESUME                   0

  1           RETURN_CONST             0 ((1, 2, 3, 'x'))

Expression: [1, 2, 3, 'x']
  0           RESUME                   0

  1           BUILD_LIST               0
              LOAD_CONST               0 ((1, 2, 3, 'x'))
              LIST_EXTEND              1
              RETURN_VALUE

Expression: (len, 1, 2)
  0           RESUME                   0

  1           LOAD_NAME                0 (len)
              LOAD_CONST               0 (1)
              LOAD_CONST               1 (2)
              BUILD_TUPLE              3
              RETURN_VALUE

Expression: [len, 1, 2]
  0           RESUME                   0

  1           LOAD_NAME                0 (len)
              LOAD_CONST               0 (1)
              LOAD_CONST               1 (2)
              BUILD_LIST               3
              RETURN_VALUE

Expression: ([1, 2], 3, 4)
  0           RESUME                   0

  1           LOAD_CO

### Solution 1

A tuple made entirely from compile-time constants, such as `(1, 2, 3, 'x')`, can usually be represented in the code object's constants table and loaded with a single `LOAD_CONST`.

A list literal such as `[1, 2, 3, 'x']` cannot be loaded as one immutable constant because lists are mutable. CPython must create a fresh list object at runtime, so a list-building operation is required.

For `(len, 1, 2)`, `len` is a name lookup, not a compile-time constant. The tuple cannot be fully precomputed, so CPython must load `len`, load the constants, and then build the tuple at runtime.

For `[len, 1, 2]`, a runtime build is also required because the object is a list and because `len` must be resolved at runtime.

For `([1, 2], 3, 4)`, the outer object is a tuple, but the tuple contains a list. The inner list must be constructed at runtime, and therefore the outer tuple must also be constructed at runtime.

**Key idea:** tuple immutability allows CPython to reuse constant tuple objects when all elements are constants. List mutability prevents this optimization.

## Problem 2 — The Tuple That Is Immutable but Still Changes

**Difficulty:** Advanced

Analyze this code without running it first:

```python
t = ([10, 20], {'a': 1})
u = tuple(t)
v = t[:]

t[0].append(30)
t[1]['b'] = 2

result = (t is u, t is v, u, v)
```

Answer:

1. Are `t`, `u`, and `v` the same tuple object?
2. Do `u` and `v` observe the mutations?
3. What does this prove about tuple immutability?


In [3]:
t = ([10, 20], {'a': 1})
u = tuple(t)
v = t[:]

t[0].append(30)
t[1]['b'] = 2

result = (t is u, t is v, u, v)
result


(True,
 True,
 ([10, 20, 30], {'a': 1, 'b': 2}),
 ([10, 20, 30], {'a': 1, 'b': 2}))

### Solution 2

`tuple(t)` returns `t` itself when `t` is already an exact tuple, so `t is u` is `True`.

A full tuple slice `t[:]` also commonly returns the original tuple object, so `t is v` is also expected to be `True` in CPython.

Both `u` and `v` observe the mutations because the tuple stores references to objects. The tuple prevents rebinding its slots, but it does not recursively freeze the objects referenced by those slots.

After mutation, the tuple still has the same two references, but the referenced list and dictionary have changed.

**Important distinction:**

- Tuple immutability means the tuple's item references cannot be changed.
- It does **not** mean the referenced objects are immutable.
- A tuple containing mutable objects is not deeply immutable.


## Problem 3 — Hashability Trap: When Can a Tuple Be a Dictionary Key?

**Difficulty:** Advanced

For each object below, predict whether it can be used as a dictionary key:

```python
a = (1, 2, 3)
b = ([1, 2], 3)
c = ((1, 2), (3, 4))
d = (frozenset({1, 2}), 'x')
e = (set([1, 2]), 'x')
```

Then write a function `is_hashable(obj)` that returns `True` only if `hash(obj)` succeeds.

In [4]:
def is_hashable(obj):
    try:
        hash(obj)
    except TypeError:
        return False
    else:
        return True

objects = {
    'a': (1, 2, 3),
    'b': ([1, 2], 3),
    'c': ((1, 2), (3, 4)),
    'd': (frozenset({1, 2}), 'x'),
    'e': (set([1, 2]), 'x'),
}

{name: is_hashable(value) for name, value in objects.items()}


{'a': True, 'b': False, 'c': True, 'd': True, 'e': False}

### Solution 3

`a` is hashable because all of its elements are hashable integers.

`b` is not hashable because it contains a list, and lists are unhashable.

`c` is hashable because it contains tuples of integers, and all nested elements are hashable.

`d` is hashable because `frozenset` is immutable and hashable when its elements are hashable.

`e` is not hashable because it contains a mutable `set`, which is unhashable.

**Rule:** a tuple is hashable only if every element inside it is hashable. Tuple immutability alone is not enough.

## Problem 4 — Measure Memory Growth and Infer Over-Allocation

**Difficulty:** Advanced

Write a function `growth_profile(factory, n)` that creates containers of length `0` through `n` using `factory(range(k))`, and records their sizes using `sys.getsizeof`.

Use it to compare `tuple` and `list` for `n = 80`.

Answer:

1. Which one grows more regularly?
2. Which one sometimes jumps by more than one pointer-width?
3. Why does this happen?


In [5]:
def growth_profile(factory, n=80):
    rows = []
    previous = None
    for k in range(n + 1):
        obj = factory(range(k))
        size = sys.getsizeof(obj)
        delta = None if previous is None else size - previous
        rows.append((k, size, delta))
        previous = size
    return rows

tuple_profile = growth_profile(tuple, 80)
list_profile = growth_profile(list, 80)

print('Tuple profile, first 12 rows:')
for row in tuple_profile[:12]:
    print(row)

print('\nList profile, first 20 rows:')
for row in list_profile[:20]:
    print(row)


Tuple profile, first 12 rows:
(0, 40, None)
(1, 48, 8)
(2, 56, 8)
(3, 64, 8)
(4, 72, 8)
(5, 80, 8)
(6, 88, 8)
(7, 96, 8)
(8, 104, 8)
(9, 112, 8)
(10, 120, 8)
(11, 128, 8)

List profile, first 20 rows:
(0, 56, None)
(1, 72, 16)
(2, 72, 0)
(3, 88, 16)
(4, 88, 0)
(5, 104, 16)
(6, 104, 0)
(7, 120, 16)
(8, 120, 0)
(9, 136, 16)
(10, 136, 0)
(11, 152, 16)
(12, 152, 0)
(13, 168, 16)
(14, 168, 0)
(15, 184, 16)
(16, 184, 0)
(17, 200, 16)
(18, 200, 0)
(19, 216, 16)


### Solution 4

Tuples usually grow regularly because a tuple has a fixed length after creation. Its storage can be allocated precisely for the number of element references it needs.

Lists sometimes jump by more than one pointer-width because lists over-allocate. A list often reserves more capacity than its current length so that future appends do not require resizing every time.

This is a classic dynamic-array tradeoff:

- **More memory:** lists may reserve extra slots.
- **Better append performance:** repeated append operations are amortized efficient.
- **Less predictable size growth:** memory increases happen in jumps rather than one element at a time.

Tuples do not need append-friendly spare capacity because their length is fixed.

## Problem 5 — Design a Reliable Benchmark for Tuple vs List Literals

**Difficulty:** Advanced

A student writes this benchmark:

```python
timeit('(1, 2, 3, 4, 5)', number=10_000_000)
timeit('[1, 2, 3, 4, 5]', number=10_000_000)
```

They conclude: "Tuples are always faster than lists."

Critique the conclusion. Then build a better benchmark that compares:

1. Constant tuple literal creation.
2. Constant list literal creation.
3. Tuple creation with a runtime name lookup.
4. List creation with a runtime name lookup.
5. Tuple/list conversion from existing objects.


In [6]:
x = object()
existing_tuple = tuple(range(20))
existing_list = list(range(20))

benchmarks = {
    'constant tuple literal': "(1, 2, 3, 4, 5)",
    'constant list literal': "[1, 2, 3, 4, 5]",
    'runtime-name tuple': "(x, 1, 2, 3, 4)",
    'runtime-name list': "[x, 1, 2, 3, 4]",
    'tuple(existing_tuple)': "tuple(existing_tuple)",
    'list(existing_list)': "list(existing_list)",
    'tuple(existing_list)': "tuple(existing_list)",
    'list(existing_tuple)': "list(existing_tuple)",
}

for label, stmt in benchmarks.items():
    elapsed = timeit(stmt, globals=globals(), number=1_000_000)
    print(f"{label:24s} {elapsed:.6f}s")


constant tuple literal   0.018047s
constant list literal    0.069358s
runtime-name tuple       0.051908s
runtime-name list        0.059587s
tuple(existing_tuple)    0.019399s
list(existing_list)      0.141947s
tuple(existing_list)     0.065556s
list(existing_tuple)     0.127210s


### Solution 5

The student's conclusion is too broad.

The constant tuple literal benchmark benefits from compile-time constant folding. CPython can often load the entire tuple as a single constant, while the list must be freshly built at runtime because it is mutable.

That does not prove tuples are always faster in all operations. It proves that a particular tuple literal can be loaded very efficiently.

A better benchmark separates several cases:

- Loading a fully constant tuple literal.
- Building a list literal.
- Building containers containing runtime values.
- Converting an existing tuple with `tuple(existing_tuple)`, which can return the same object.
- Converting an existing list with `list(existing_list)`, which must create a shallow copy.

**Best practice:** benchmark the exact operation you care about, isolate variables, avoid overgeneralizing, and inspect bytecode when compiler optimizations may be involved.

## Problem 6 — API Design: Prevent Accidental Mutation of Internal State

**Difficulty:** Advanced

You are designing a class that stores a sequence of registered event handlers. You want users to inspect the handlers but not mutate the internal registry directly.

The first implementation is unsafe:

```python
class Registry:
    def __init__(self):
        self._handlers = []

    def add(self, handler):
        self._handlers.append(handler)

    @property
    def handlers(self):
        return self._handlers
```

Explain the bug and rewrite the property so callers cannot append to the internal list through the returned object.

In [7]:
class Registry:
    def __init__(self):
        self._handlers = []

    def add(self, handler):
        self._handlers.append(handler)

    @property
    def handlers(self):
        # Return an immutable snapshot of the current handler references.
        return tuple(self._handlers)


def h1():
    return 'h1'

r = Registry()
r.add(h1)
exposed = r.handlers
print(exposed)

try:
    exposed.append(lambda: 'bad')
except AttributeError as exc:
    print(type(exc).__name__, exc)


(<function h1 at 0x00000287FFF9CFE0>,)
AttributeError 'tuple' object has no attribute 'append'


### Solution 6

The unsafe version returns the actual internal list. A caller can mutate it directly:

```python
registry.handlers.append(bad_handler)
```

That bypasses the `add` method and breaks encapsulation.

Returning `tuple(self._handlers)` gives callers an immutable snapshot. They can inspect and iterate over the handlers, but they cannot append, delete, or reorder the returned container.

This does not make the handler objects themselves immutable. It only protects the registry's sequence structure from direct external mutation.

**API design lesson:** use tuples to communicate and enforce fixed sequence semantics at the boundary of an object.

## Problem 7 — Shallow Copy vs Deep Copy in Nested Containers

**Difficulty:** Advanced

Predict the output:

```python
original = ([1, 2], [3, 4])
shallow = tuple(original)
deep = copy.deepcopy(original)

original[0].append(99)

print(original)
print(shallow)
print(deep)
print(original is shallow)
print(original[0] is shallow[0])
print(original[0] is deep[0])
```

Then explain why `tuple(original)` may not create the protection someone expects.

In [8]:
original = ([1, 2], [3, 4])
shallow = tuple(original)
deep = copy.deepcopy(original)

original[0].append(99)

print(original)
print(shallow)
print(deep)
print(original is shallow)
print(original[0] is shallow[0])
print(original[0] is deep[0])


([1, 2, 99], [3, 4])
([1, 2, 99], [3, 4])
([1, 2], [3, 4])
True
True
False


### Solution 7

`tuple(original)` returns the same tuple object when `original` is already a tuple. Even when tuple construction does create a new tuple from another iterable, it is still shallow: it copies references, not the objects behind those references.

Therefore, `shallow` observes the mutation to the nested list. `deep` does not, because `copy.deepcopy` recursively creates independent nested objects.

Expected interpretation:

- `original` contains `[1, 2, 99]` as its first nested list.
- `shallow` sees the same nested list.
- `deep` still contains `[1, 2]`.
- `original is shallow` is expected to be `True` here because `tuple(original)` returns the same tuple.
- `original[0] is shallow[0]` is `True`.
- `original[0] is deep[0]` is `False`.

**Best practice:** if you need deep immutability, converting the outer container to a tuple is insufficient. You must also freeze or copy the nested mutable objects.

## Problem 8 — Implement a Recursive Freezer

**Difficulty:** Advanced

Write a function `freeze(obj)` that recursively converts:

- lists to tuples,
- tuples to tuples containing frozen elements,
- sets to frozensets,
- dictionaries to tuples of sorted key-value pairs, where both keys and values are frozen.

Then show that the result can be used as a dictionary key for ordinary nested data made from strings, numbers, lists, tuples, sets, and dictionaries.

In [9]:
def freeze(obj):
    if isinstance(obj, list):
        return tuple(freeze(item) for item in obj)
    if isinstance(obj, tuple):
        return tuple(freeze(item) for item in obj)
    if isinstance(obj, set):
        return frozenset(freeze(item) for item in obj)
    if isinstance(obj, dict):
        return tuple(sorted((freeze(key), freeze(value)) for key, value in obj.items()))
    return obj

data = {
    'name': 'alpha',
    'scores': [10, 20, 30],
    'tags': {'python', 'containers'},
    'meta': {'active': True, 'level': 3},
}

frozen = freeze(data)
cache = {frozen: 'cached result'}

frozen, cache[frozen]


((('meta', (('active', True), ('level', 3))),
  ('name', 'alpha'),
  ('scores', (10, 20, 30)),
  ('tags', frozenset({'containers', 'python'}))),
 'cached result')

### Solution 8

The function recursively replaces mutable containers with immutable equivalents:

- `list` becomes `tuple`.
- `tuple` stays a tuple but its elements are recursively frozen.
- `set` becomes `frozenset`.
- `dict` becomes a sorted tuple of frozen key-value pairs.

Sorting dictionary items gives a deterministic representation independent of insertion order.

This approach is useful for cache keys and memoization, but it has limits:

- It assumes keys become mutually comparable after freezing, because `sorted` needs ordering.
- It does not handle arbitrary custom mutable objects.
- It does not handle cyclic data structures.

For production use, you would need extra handling for non-comparable keys and cycles.

## Problem 9 — Amortized Append: Detect Resize Events

**Difficulty:** Advanced

Build a function `list_resize_events(n)` that appends integers to a list and records only the moments when `sys.getsizeof` changes.

Then use the output to explain why list append is efficient on average even though occasional resizes are expensive.

In [10]:
def list_resize_events(n):
    items = []
    previous_size = sys.getsizeof(items)
    events = []
    for i in range(n):
        items.append(i)
        size = sys.getsizeof(items)
        if size != previous_size:
            events.append({
                'length_after_append': len(items),
                'old_size': previous_size,
                'new_size': size,
                'delta': size - previous_size,
            })
            previous_size = size
    return events

events = list_resize_events(300)
events[:15]


[{'length_after_append': 1, 'old_size': 56, 'new_size': 88, 'delta': 32},
 {'length_after_append': 5, 'old_size': 88, 'new_size': 120, 'delta': 32},
 {'length_after_append': 9, 'old_size': 120, 'new_size': 184, 'delta': 64},
 {'length_after_append': 17, 'old_size': 184, 'new_size': 248, 'delta': 64},
 {'length_after_append': 25, 'old_size': 248, 'new_size': 312, 'delta': 64},
 {'length_after_append': 33, 'old_size': 312, 'new_size': 376, 'delta': 64},
 {'length_after_append': 41, 'old_size': 376, 'new_size': 472, 'delta': 96},
 {'length_after_append': 53, 'old_size': 472, 'new_size': 568, 'delta': 96},
 {'length_after_append': 65, 'old_size': 568, 'new_size': 664, 'delta': 96},
 {'length_after_append': 77, 'old_size': 664, 'new_size': 792, 'delta': 128},
 {'length_after_append': 93, 'old_size': 792, 'new_size': 920, 'delta': 128},
 {'length_after_append': 109, 'old_size': 920, 'new_size': 1080, 'delta': 160},
 {'length_after_append': 129,
  'old_size': 1080,
  'new_size': 1240,
  'delt

### Solution 9

The resize events show that a list does not request new storage on every append. Instead, it grows its internal capacity in chunks.

Most appends place a new reference into already allocated space and are cheap. Occasionally, the list must allocate a larger internal array and copy references into it. That resize is more expensive, but it happens infrequently.

Across many appends, the expensive resizing work is spread out. This is why list append is usually described as amortized efficient.

**Connection to tuples:** tuples do not support append, so they do not need this growth strategy. Their storage can be exact at construction time.

## Problem 10 — Choose the Right Container in Real Designs

**Difficulty:** Advanced

For each scenario, choose `list`, `tuple`, or another type. Justify the choice.

1. A function returns the `(x, y, z)` coordinates of a point.
2. A function accumulates log messages during execution.
3. A cache key represents a user's immutable query filters.
4. A class exposes an internal sequence of plugins for read-only inspection.
5. A data pipeline repeatedly appends records and later freezes them for downstream consumers.
6. You need named, immutable fields for `host`, `port`, and `use_ssl`.


In [11]:
Endpoint = namedtuple('Endpoint', ['host', 'port', 'use_ssl'])

answers = {
    'coordinates': 'tuple',
    'log_accumulation': 'list',
    'cache_key': 'tuple or recursively frozen structure',
    'read_only_plugins': 'tuple snapshot',
    'append_then_freeze': 'list during construction, tuple when publishing',
    'named_immutable_fields': 'namedtuple or dataclass(frozen=True)',
}

endpoint = Endpoint('localhost', 5432, False)
answers, endpoint


({'coordinates': 'tuple',
  'log_accumulation': 'list',
  'cache_key': 'tuple or recursively frozen structure',
  'read_only_plugins': 'tuple snapshot',
  'append_then_freeze': 'list during construction, tuple when publishing',
  'named_immutable_fields': 'namedtuple or dataclass(frozen=True)'},
 Endpoint(host='localhost', port=5432, use_ssl=False))

### Solution 10

1. Coordinates are naturally positional and fixed-size, so a tuple is appropriate.
2. Log messages are accumulated over time, so a list is appropriate.
3. Cache keys must be hashable and stable, so use a tuple or recursively frozen structure.
4. Returning a tuple snapshot protects the internal plugin sequence from direct mutation.
5. A list is ideal while building because appends are efficient. Convert to a tuple when the result should become fixed.
6. Use `namedtuple` or a frozen dataclass when field names improve readability and the record should be immutable.

**Best practice:** choose containers based on semantics first, then performance. Use tuples for fixed structure and stability. Use lists for growth and mutation.

## Final Review Checklist

You should now be able to explain:

- Why constant tuples may load faster than lists.
- Why `tuple(existing_tuple)` can return the same object.
- Why tuple immutability is shallow.
- Why not every tuple is hashable.
- How list over-allocation supports efficient appends.
- Why memory measurements must be interpreted carefully.
- How to choose lists vs tuples in API design.
